In [66]:
import sys
sys.path.insert(0, 'src')
from run_pathway_materials import run_pathway_materials
import pandas as pd
import plotly.express as px
import os
import math
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [47]:

results_materials = run_pathway_materials('my_first_run_materials')

results_materials['F_new']                    # capacités nouvelles [Phases, Technologies]
results_materials['Material_content_year']    # demande matériau annualisée [Years, Technologies, Materials]
results_materials['Recycled_material']

[run_pathway_materials] Window 1/1 done in 116.6s
[run_pathway_materials] Total time: 116.6s


Recycled_material
Years     Technologies  Materials                   
YEAR_2020 AFC           Ag                         0
                        Al                         0
                        B                          0
                        Cd                         0
                        Co                         0
...                                              ...
YEAR_2050 WOOD_METHANOL V                          0
                        W                          0
                        Y                          0
                        Zn                         0
                        Zr                         0

[173880 rows x 1 columns]

In [48]:
material_content = (results_materials['Material_content_year']['Material_content_year']
                     .groupby(['Technologies', 'Materials']).sum() * 5)

In [52]:
periods = ['2020_2025', '2025_2030', '2030_2035', '2035_2040', '2040_2045', '2045_2050']
years = ['2025', '2030', '2035', '2040', '2045', '2050']
technologies = [t for t in results_materials['F_new'].loc['2020_2025'].index
                if any(results_materials['F_new'].loc[period].loc[t].squeeze() > 0 for period in periods)]

elec_keywords = ['PV_', 'WIND_', 'HYDRO', 'NUCLEAR', 'CCGT', 'COAL_', 'OCGT_', 'TIDAL', 'GEOTHERMAL', 'AFC', 'PAFC', 'PEMFC', 'SOFC', 'WAVE']

elec_techs = [t for t in results_materials['F_new'].loc['2020_2025'].index 
              if any(kw in t for kw in elec_keywords) and not t.startswith(('COAL_GAS', 'HYDRO_STORAGE', 'UNMINEABLE_COAL_SEAM'))]

elec_techs_positive = [t for t in elec_techs
                       if any(results_materials['F_new'].loc[period].loc[t].squeeze() > 0 for period in periods)]


In [53]:
df_plot = pd.DataFrame(
    {period: results_materials['F_new'].loc[period].loc[elec_techs_positive].squeeze() for period in periods},
    index=elec_techs_positive
)

df_melted = df_plot.T.reset_index().rename(columns={'index': 'Période'}).melt(
    id_vars='Période', var_name='Technologies', value_name='Capacité'
)

fig = px.bar(df_melted, x='Période', y='Capacité', color='Technologies', barmode='stack')
fig.update_layout(xaxis_title='Période', yaxis_title='Capacité [GW]')
fig.show()

In [54]:
from shared.utils import run_pathway

In [55]:
results_pathway = run_pathway('my_first_run')

[run_pathway] Window 1/1 done in 109.1s
[run_pathway] Total time: 109.1s


In [62]:
mcy_cu = results_materials['Material_content_year'].xs('Cu', level='Materials')
mcy_cu.groupby('Years').sum()

,Material_content_year
Years,
YEAR_2020,0.000000
YEAR_2025,466.795438
YEAR_2030,0.000000
YEAR_2035,7059.036465
YEAR_2040,461.603055
YEAR_2045,472.970388
YEAR_2050,510.329290


In [72]:
mcy = results_materials['Material_content_year']['Material_content_year']
demand = mcy.groupby(['Years', 'Materials']).sum().unstack('Materials')  # index=Years, colonnes=Materials
demand = demand.loc[:, (demand.fillna(0) != 0).any(axis=0)]  # enleve les materiaux a zero partout

materials = demand.columns.tolist()
n = len(materials)
ncols = 6
nrows = -(-n // ncols)

fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=materials)

years_x = [int(y.replace('YEAR_', '')) for y in demand.index.tolist()]

for i, material in enumerate(materials):
    row = i // ncols + 1
    col = i % ncols + 1
    fig.add_trace(
        go.Bar(x=years_x, y=demand[material].values, name=material, showlegend=False),
        row=row, col=col
    )

fig.update_layout(height=300 * nrows, title='Demande annuelle en matériau (modèle pathway + contraintes)')
fig.update_yaxes(title_text='[t/an]', col=1)
fig.update_xaxes(tickmode='array', tickvals=years_x, tickangle=45)
fig.show()
save_dir = os.path.expanduser('~/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy_infinite')
os.makedirs(save_dir, exist_ok=True)

filepath = os.path.join(save_dir, 'tot_mat_elec_infinite.png')
fig.write_image(filepath, width=250*ncols, height=300*nrows)